# Dashboard Deteksi Kebocoran Gas (SIC 8)

Notebook ini menerima data dari ESP32 (simulasi Wokwi) lewat **MQTT / EMQX Serverless** (broker gratis), lalu menampilkannya dengan **Plotly**.

**Cara pakai**
1. Jalankan simulasi Wokwi sampai Serial Monitor menampilkan `[MQTT] Terhubung ke broker`.
2. Jalankan sel 1 sampai 4 secara berurutan. Sel 2 akan meminta password EMQX, yang tidak disimpan di notebook.
3. Sel 4 mengumpulkan data selama `COLLECT_SECONDS`. Selama itu, ubah kondisi di Wokwi (geser slider gas, picu PIR, tekan tombol).
4. Jalankan sel 5 untuk menampilkan grafik dan tabel event.
5. Ingin data lebih banyak? Jalankan ulang sel 4 (data lama tetap disimpan), lalu sel 5.

**Topic yang dibaca**
- `sic8/gasleak/status`: JSON `{"gas", "raw", "motion", "alarm", "pos", "event"}`
- `sic8/gasleak/availability`: `online` / `offline`


In [ ]:
# 1. Install the MQTT client (pandas and plotly are already available in Colab)
!pip install -q paho-mqtt

: 

In [ ]:
# 2. Broker settings: fill in HOST and USER; the password is asked at runtime so it is never saved here
import getpass

MQTT_HOST = "ISI-ALAMAT-DEPLOYMENT.emqxsl.com"
MQTT_PORT = 8883                 # MQTT over TLS
MQTT_USER = "ISI-USERNAME"
MQTT_PASS = getpass.getpass("Password EMQX: ")

TOPIC = "sic8/gasleak/#"         # '#' = all topics under sic8/gasleak/
COLLECT_SECONDS = 60             # how long cell 4 listens for data

In [ ]:
# 3. Storage + message parser: every status message becomes one row of the table
import json
from datetime import datetime

import pandas as pd

rows = []                                         # collected status messages
availability = {"state": "unknown", "time": None} # last online/offline seen


def handle_message(topic, payload, now=None):
    """Store one MQTT message: availability updates the device state, status becomes a row."""
    now = now or datetime.now()
    text = payload.decode("utf-8", errors="replace")
    if topic.endswith("/availability"):
        availability.update(state=text, time=now)
        return
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        print("Payload bukan JSON, dilewati:", text)
        return
    data["time"] = now
    rows.append(data)

In [ ]:
# 4. Connect to the EMQX Serverless broker and collect messages for COLLECT_SECONDS
import time

import paho.mqtt.client as mqtt


def on_connect(client, userdata, flags, reason_code, properties):
    """Subscribe once the broker accepts the connection."""
    if reason_code.is_failure:
        print("Gagal terhubung:", reason_code)
        return
    print("Terhubung ke broker, subscribe", TOPIC)
    client.subscribe(TOPIC)


def on_message(client, userdata, msg):
    """Hand every incoming message to the parser from cell 3."""
    handle_message(msg.topic, msg.payload)


client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.username_pw_set(MQTT_USER, MQTT_PASS)
client.tls_set()                   # TLS with the system CA list (EMQX Serverless requires TLS)
client.on_connect = on_connect
client.on_message = on_message

client.connect(MQTT_HOST, MQTT_PORT)
client.loop_start()                # network loop runs in a background thread
print(f"Mengumpulkan data selama {COLLECT_SECONDS} detik...")
time.sleep(COLLECT_SECONDS)
client.loop_stop()
client.disconnect()

print(f"Total pesan status: {len(rows)} | status perangkat: {availability['state']}")

In [ ]:
# 5. Dashboard: gas trend + status timelines + event table
import plotly.graph_objects as go
from plotly.subplots import make_subplots

STATUS_ROWS = [("gas", "Gas terdeteksi"), ("alarm", "Alarm aktif"), ("motion", "Ada orang")]


def build_dashboard(df, device_state):
    """One figure: analog gas level on top, one 0/1 step chart per status below it."""
    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.06,
        row_heights=[0.4, 0.2, 0.2, 0.2],
        subplot_titles=["Kadar gas (AOUT, 0-4095)"] + [label for _, label in STATUS_ROWS],
    )
    fig.add_trace(go.Scatter(x=df["time"], y=df["raw"], mode="lines+markers", name="Kadar gas"), row=1, col=1)
    for i, (col, label) in enumerate(STATUS_ROWS, start=2):
        # line_shape="hv" draws a step: the value holds until the next message
        fig.add_trace(go.Scatter(x=df["time"], y=df[col], mode="lines", line_shape="hv", name=label), row=i, col=1)
        fig.update_yaxes(range=[-0.1, 1.1], tickvals=[0, 1], ticktext=["tidak", "ya"], row=i, col=1)
    latest = df.iloc[-1]
    fig.update_layout(
        height=750, showlegend=False,
        title=f"Gas Leak Monitor | perangkat: {device_state} | alarm: {'AKTIF' if latest['alarm'] else 'normal'}",
    )
    return fig


if not rows:
    print("Belum ada data. Pastikan simulasi Wokwi berjalan, lalu jalankan ulang sel 4.")
else:
    df = pd.DataFrame(rows)
    build_dashboard(df, availability["state"]).show()
    # Event log: everything except the periodic heartbeat
    display(df[df["event"] != "heartbeat"][["time", "event", "gas", "alarm", "motion", "raw", "pos"]])